In [ ]:
def is_variable(x):
    return isinstance(x, str) and x.startswith('?')


def substitute(expr, subst):
    if is_variable(expr) and expr in subst:
        return substitute(subst[expr], subst)

    if isinstance(expr, tuple):
        return tuple(substitute(item, subst) for item in expr)

    return expr


def unify(x, y, subst=None):
    if subst is None:
        subst = {}

    x = substitute(x, subst)
    y = substitute(y, subst)

    if x == y:
        return subst

    if is_variable(x):
        subst[x] = y
        return subst

    if is_variable(y):
        subst[y] = x
        return subst

    if isinstance(x, tuple) and isinstance(y, tuple):
        if len(x) != len(y):
            return None

        for a, b in zip(x, y):
            subst = unify(a, b, subst)
            if subst is None:
                return None

        return subst

    return None


def match_premises(premises, facts, subst=None):
    if subst is None:
        subst = {}

    if not premises:
        return [subst]

    results = []

    first = substitute(premises[0], subst)

    for fact in facts:
        new_subst = unify(first, fact, subst.copy())

        if new_subst is not None:
            results.extend(
                match_premises(
                    premises[1:],
                    facts,
                    new_subst
                )
            )

    return results


def forward_chaining(KB, rules, query):

    # Check whether query is already in KB
    for fact in KB:
        result = unify(query, fact)
        if result is not None:
            return result

    while True:
        new = set()

        # Check every rule
        for premises, conclusion in rules:

            substitutions = match_premises(
                premises, KB
            )

            for theta in substitutions:

                q = substitute(conclusion, theta)

                if q not in KB and q not in new:

                    # Check whether inferred fact matches query
                    result = unify(query, q)

                    if result is not None:
                        return result

                    new.add(q)

        # Stop if no new facts are inferred
        if not new:
            return False

        # Add new facts to knowledge base
        KB.update(new)


# MAIN PROGRAM

# Knowledge Base (Facts)
KB = {
    ('Parent', 'John', 'Mary'),
    ('Parent', 'Mary', 'Sam')
}

# Rules
rules = [
    (
        [
            ('Parent', '?x', '?y'),
            ('Parent', '?y', '?z')
        ],
        ('Grandparent', '?x', '?z')
    )
]

# Query
query = ('Grandparent', 'John', '?who')

print("FORWARD CHAINING ALGORITHM")

result = forward_chaining(KB, rules, query)

if result:
    print("Query Successful")
    print("Substitution:", result)
else:
    print("Query Failed")

